# Lab W9: Multimodal Ablation


**Pendamping Bab W9.**\n\nTujuan: mengimplementasikan **late fusion** pada dataset multimodal sintetis, menjalankan **per-modality ablation protocol** (7 kondisi), mendeteksi **ignored-modality problem**, dan menerapkan **modality dropout** sebagai solusi.\n\nDataset: synthetic face image features (20-dim) + sensor readings (15-dim).\nTarget: continuous stress level (regression).\n**Estimasi:** 2-3 jam.


## 1. Setup & Data


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Synthetic multimodal dataset: "face image features" + "sensor readings"
# Target: continuous stress level (regression)
# Realistic: both modalities contribute, but unequally

N = 5000
N_FACE = 20   # face feature dim
N_SENSOR = 15 # sensor feature dim

# Face features: moderate predictive power
face_w = torch.randn(N_FACE) * 0.4
face_X = torch.randn(N, N_FACE)
face_signal = face_X @ face_w

# Sensor features: stronger predictive power
sensor_w = torch.randn(N_SENSOR) * 0.6
sensor_X = torch.randn(N, N_SENSOR)
sensor_signal = sensor_X @ sensor_w

# Target: weighted combination + noise
y = face_signal + sensor_signal + torch.randn(N) * 0.2
y = (y - y.mean()) / y.std()  # normalize

# Split
n_tr, n_va = 3000, 1000
Xf_tr, Xf_va = face_X[:n_tr], face_X[n_tr:n_tr+n_va]
Xs_tr, Xs_va = sensor_X[:n_tr], sensor_X[n_tr:n_tr+n_va]
y_tr, y_va = y[:n_tr], y[n_tr:n_tr+n_va]

dl_tr = DataLoader(TensorDataset(Xf_tr, Xs_tr, y_tr), 64, shuffle=True)
dl_va = DataLoader(TensorDataset(Xf_va, Xs_va, y_va), 64, shuffle=False)

print(f"Train: face {Xf_tr.shape}, sensor {Xs_tr.shape}")
print(f"Val:   face {Xf_va.shape}, sensor {Xs_va.shape}")
print(f"y mean/std: {y_tr.mean():.3f}/{y_tr.std():.3f}")

## 2. Late Fusion Model + Baseline


In [ ]:
class LateFusionModel(nn.Module):
    def __init__(self, d_face=20, d_sensor=15, d_hidden=32, modality_dropout=0.0):
        super().__init__()
        self.enc_face = nn.Sequential(
            nn.Linear(d_face, d_hidden), nn.ReLU(), nn.Dropout(modality_dropout))
        self.enc_sensor = nn.Sequential(
            nn.Linear(d_sensor, d_hidden), nn.ReLU(), nn.Dropout(modality_dropout))
        self.head = nn.Linear(d_hidden * 2, 1)

    def forward(self, face, sensor):
        h_f = self.enc_face(face)
        h_s = self.enc_sensor(sensor)
        fused = torch.cat([h_f, h_s], dim=1)
        return self.head(fused).squeeze(-1)

    def encode_face(self, face):
        return self.enc_face(face)

    def encode_sensor(self, sensor):
        return self.enc_sensor(sensor)

def train_model(model, dl_tr, dl_va, epochs=30, lr=5e-4):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.MSELoss()
    history = {"train_loss": [], "val_mae": []}
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for Xf, Xs, yb in dl_tr:
            Xf, Xs, yb = Xf.to(DEVICE), Xs.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(Xf, Xs), yb)
            loss.backward()
            opt.step()
            running += loss.item() * len(Xf)
        # Eval
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for Xf, Xs, yb in dl_va:
                Xf, Xs, yb = Xf.to(DEVICE), Xs.to(DEVICE), yb.to(DEVICE)
                pred = model(Xf, Xs)
                preds.append(pred.cpu())
                trues.append(yb.cpu())
        preds = torch.cat(preds)
        trues = torch.cat(trues)
        mae = (preds - trues).abs().mean().item()
        history["train_loss"].append(running / len(dl_tr.dataset))
        history["val_mae"].append(mae)
    return history

def evaluate(model, dl_va):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xf, Xs, yb in dl_va:
            Xf, Xs, yb = Xf.to(DEVICE), Xs.to(DEVICE), yb.to(DEVICE)
            pred = model(Xf, Xs)
            preds.append(pred.cpu())
            trues.append(yb.cpu())
    preds = torch.cat(preds)
    trues = torch.cat(trues)
    mae = (preds - trues).abs().mean().item()
    return mae

In [ ]:
model = LateFusionModel().to(DEVICE)
hist = train_model(model, dl_tr, dl_va)

plt.figure(figsize=(10, 3))
plt.subplot(121)
plt.plot(hist["train_loss"]); plt.title("Train Loss (MSE)"); plt.xlabel("epoch"); plt.grid(alpha=0.3)
plt.subplot(122)
plt.plot(hist["val_mae"]); plt.title("Val MAE"); plt.xlabel("epoch"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

mae_fusion = evaluate(model, dl_va)
print(f"Late Fusion (both modalities) - Val MAE: {mae_fusion:.4f}")

## 3. Per-Modality Ablation (7 conditions)\n\n1. Face only (sensor=0)\n2. Sensor only (face=0)\n3. Fusion (both)\n4. Random face (face di-shuffle saat eval)\n5. Random sensor (sensor di-shuffle saat eval)\n6. Random both\n\nBandingkan: jika face only ≈ random face → face diabaikan (ignored-modality).


In [ ]:
# 7 ablation conditions
face_dummy = torch.zeros(N_FACE)  # menandai "face tidak ada"
sensor_dummy = torch.zeros(N_SENSOR)

print("=== ABLATION RESULTS ===\n")
results = {}

# 1. Face-only (sensor di-zero-kan)
model_face = LateFusionModel().to(DEVICE)
train_model(model_face, dl_tr, dl_va)
mae_fo = evaluate(model_face, dl_va)
results["Face only"] = mae_fo

# 2. Sensor-only (face di-zero-kan)
model_sensor = LateFusionModel().to(DEVICE)
train_model(model_sensor, dl_tr, dl_va)
mae_so = evaluate(model_sensor, dl_va)
results["Sensor only"] = mae_so

# 3. Random face (shuffle face features during eval)
model_rf = LateFusionModel().to(DEVICE)
train_model(model_rf, dl_tr, dl_va)
# evaluate dengan face di-random shuffle
model_rf.eval()
preds, trues = [], []
with torch.no_grad():
    for Xf, Xs, yb in dl_va:
        Xf_shuf = Xf[torch.randperm(len(Xf))]
        Xf_shuf, Xs, yb = Xf_shuf.to(DEVICE), Xs.to(DEVICE), yb.to(DEVICE)
        pred = model_rf(Xf_shuf, Xs)
        preds.append(pred.cpu()); trues.append(yb.cpu())
mae_rf = (torch.cat(preds) - torch.cat(trues)).abs().mean().item()
results["Random face"] = mae_rf

# 4. Random sensor (shuffle sensor features during eval)
model_rs = LateFusionModel().to(DEVICE)
train_model(model_rs, dl_tr, dl_va)
model_rs.eval()
preds, trues = [], []
with torch.no_grad():
    for Xf, Xs, yb in dl_va:
        Xs_shuf = Xs[torch.randperm(len(Xs))]
        Xf, Xs_shuf, yb = Xf.to(DEVICE), Xs_shuf.to(DEVICE), yb.to(DEVICE)
        pred = model_rs(Xf, Xs_shuf)
        preds.append(pred.cpu()); trues.append(yb.cpu())
mae_rs = (torch.cat(preds) - torch.cat(trues)).abs().mean().item()
results["Random sensor"] = mae_rs

# 5. Both random
model_br = LateFusionModel().to(DEVICE)
train_model(model_br, dl_tr, dl_va)
model_br.eval()
preds, trues = [], []
with torch.no_grad():
    for Xf, Xs, yb in dl_va:
        Xf_shuf = Xf[torch.randperm(len(Xf))]
        Xs_shuf = Xs[torch.randperm(len(Xs))]
        Xf_shuf, Xs_shuf, yb = Xf_shuf.to(DEVICE), Xs_shuf.to(DEVICE), yb.to(DEVICE)
        pred = model_br(Xf_shuf, Xs_shuf)
        preds.append(pred.cpu()); trues.append(yb.cpu())
mae_br = (torch.cat(preds) - torch.cat(trues)).abs().mean().item()
results["Random both"] = mae_br

# Print results table
print(f"{'Condition':20s} {'Val MAE':>8s} {'vs Fusion':>10s}")
print("-" * 40)
for cond, mae in sorted(results.items(), key=lambda x: x[1]):
    delta = mae - mae_fusion
    print(f"{cond:20s} {mae:8.4f} {delta:+10.4f}")
print(f"{'Fusion (both)':20s} {mae_fusion:8.4f} {'0.0000':>10s}")
print(f"\n{'Baseline (random)':20s} {mae_br:8.4f} {mae_br-mae_fusion:+10.4f}")

## 4. Diagnostik Visual


In [ ]:
# Visualisasi
conds = list(results.keys()) + ["Fusion (both)"]
vals = list(results.values()) + [mae_fusion]
colors = ["#e74c3c", "#3498db", "#e67e22", "#9b59b6", "#95a5a6", "#2ecc71"]

plt.figure(figsize=(10, 5))
bars = plt.bar(range(len(vals)), vals, color=colors[:len(vals)])
plt.xticks(range(len(vals)), conds, rotation=15, ha="right")
plt.ylabel("Val MAE")
plt.title("Multimodal Ablation: Per-Modality Contribution")
plt.axhline(y=mae_br, color="gray", linestyle="--", label=f"Random baseline ({mae_br:.3f})")
plt.legend(); plt.grid(alpha=0.3, axis="y")

for bar, v in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

print("\n=== DIAGNOSIS ===")
print("Jika 'Face only' ~ 'Random face' -> face modality diabaikan (ignored-modality)")
print("Jika 'Sensor only' ~ 'Random sensor' -> sensor modality diabaikan")
print("Jika 'Random both' ~ 'Random face' -> hanya sensor yg dipakai")
print()
if abs(results["Face only"] - results["Random face"]) < 0.05:
    print(">>> KECURIGAAN: Face only ~ Random face -> face modality IGNORED!")
elif abs(results["Sensor only"] - results["Random sensor"]) < 0.05:
    print(">>> KECURIGAAN: Sensor only ~ Random sensor -> sensor modality IGNORED!")
else:
    print(">>> Kedua modality berkontribusi. Tidak ada ignored-modality problem.")
    print("    Tapi tetap perlu modality dropout untuk robustness.")

## 5. Modality Dropout\n\nTraining dengan random zero-out salah satu modality (15% chance). Ini memaksa model belajar representasi yang berguna dari tiap modality secara mandiri.


In [ ]:
# Modality Dropout: selama training, kadang-kadang zero-kan satu modality
# Ini memaksa model untuk tidak bergantung penuh pada satu modality saja.

model_md = LateFusionModel(modality_dropout=0.2).to(DEVICE)

# Training loop dengan modality dropout (random zero-out)
opt = optim.AdamW(model_md.parameters(), lr=5e-4, weight_decay=1e-4)
crit = nn.MSELoss()
history_md = []
for epoch in range(40):
    model_md.train()
    running = 0.0
    for Xf, Xs, yb in dl_tr:
        Xf, Xs, yb = Xf.to(DEVICE), Xs.to(DEVICE), yb.to(DEVICE)
        # Random modality dropout: 15% chance zero-out face, 15% chance zero-out sensor
        if torch.rand(1).item() < 0.15:
            Xf = torch.zeros_like(Xf)
        if torch.rand(1).item() < 0.15:
            Xs = torch.zeros_like(Xs)
        opt.zero_grad()
        loss = crit(model_md(Xf, Xs), yb)
        loss.backward()
        opt.step()
        running += loss.item() * len(Xf)
    history_md.append(running / len(dl_tr.dataset))

# Evaluate with each modality alone
model_md.eval()
def eval_modality(model, dl_va, use_face=True):
    preds, trues = [], []
    with torch.no_grad():
        for Xf, Xs, yb in dl_va:
            Xf, Xs, yb = Xf.to(DEVICE), Xs.to(DEVICE), yb.to(DEVICE)
            if not use_face:
                Xf = torch.zeros_like(Xf)
            else:
                Xs = torch.zeros_like(Xs)
            pred = model(Xf, Xs)
            preds.append(pred.cpu()); trues.append(yb.cpu())
    return (torch.cat(preds) - torch.cat(trues)).abs().mean().item()

md_face = eval_modality(model_md, dl_va, use_face=True)
md_sensor = eval_modality(model_md, dl_va, use_face=False)

print("=== Modality Dropout (0.2) Results ===")
print(f"Face only (model trained with MD):  {md_face:.4f}")
print(f"Sensor only (model trained with MD): {md_sensor:.4f}")
print()
print("Bandingkan dengan model tanpa MD:")
print(f"Face only:   {results['Face only']:.4f} -> {md_face:.4f}")
print(f"Sensor only: {results['Sensor only']:.4f} -> {md_sensor:.4f}")
print()
if md_face < results['Face only'] and md_sensor < results['Sensor only']:
    print(">>> Modality dropout meningkatkan performa single-modality. Berhasil!")
elif md_face < results['Face only']:
    print(">>> Face modality jadi lebih robust.")
elif md_sensor < results['Sensor only']:
    print(">>> Sensor modality jadi lebih robust.")

plt.figure(figsize=(8, 3))
plt.plot(history_md); plt.title("Modality Dropout Training Loss")
plt.xlabel("epoch"); plt.grid(alpha=0.3); plt.show()

## 5. Refleksi

1. **Ignored modality.** Jika face modality diabaikan (face only ≈ random face), apa artinya dalam praktiknya? Apakah Anda bisa deploy model yang hanya menggunakan sensor, atau harus memperbaiki representasi face?

2. **Trade-off modality dropout.** Modality dropout meningkatkan robustness tapi menambah epoch training. Bagaimana Anda memilih dropout rate yang tepat untuk dataset multimodal dengan 5 modality?

3. **Koneksi dengan Capstone.** Apakah dataset Capstone Anda multimodal? Jika ya, apa modality-modality-nya, dan bagaimana rancangan ablation Anda? Jika tidak, modality apa yang paling mungkin ditambahkan jika proyek diperluas?
